# Đánh giá kiến trúc backend trên Kaggle CPU

Chọn **Accelerator: None**, bật Internet để chuẩn bị dependency. Thêm Dataset riêng chứa `cinema-phase2-source.tar.gz` được tạo bằng công cụ bundle. Chạy các ô theo thứ tự trong cùng phiên CPU. Kết quả smoke không thay thế ma trận đo chính gồm 144 lượt.


In [ ]:
from pathlib import Path
import hashlib, os, subprocess, sys, tarfile
archives = list(Path('/kaggle/input').rglob('cinema-phase2-source.tar.gz'))
assert len(archives) == 1, 'Cần đúng một Dataset chứa cinema-phase2-source.tar.gz'
archive = archives[0]
bundle_sha = hashlib.sha256(archive.read_bytes()).hexdigest()
repo = Path('/kaggle/working') / ('cinema-phase2-' + bundle_sha[:12])
if not repo.exists():
    repo.mkdir()
    with tarfile.open(archive) as source:
        source.extractall(repo, filter='data')
print('Source checksum:', bundle_sha)
print('Repository:', repo)


## Chuẩn bị môi trường

Ô này cài PostgreSQL 16, Redis 7 và môi trường Python 3.12 với dependency đã khóa. Trình cài chỉ chạy trong Kaggle; các dịch vụ thử nghiệm được khởi động riêng khi kiểm chứng hoặc đo tải.


In [ ]:
sys.path.insert(0, str(repo))
from experiments.phase2.notebook import run_checked
run_checked([sys.executable, str(repo / 'experiments/phase2/bootstrap.py'), '--install'], cwd=repo)
python = '/kaggle/working/phase2-venv/bin/python'
def run(*arguments):
    run_checked([python, *arguments], cwd=repo)


## Kiểm chứng chức năng

Chạy toàn bộ kiểm thử, bao gồm PostgreSQL qua migration, cache fallback và tranh chấp giữa hai worker. Dừng tại đây nếu kiểm thử không đạt.


In [ ]:
run('-m', 'experiments.phase2.verify')


## Kiểm tra nhanh

Smoke gồm 16 lượt, 10 người dùng, warm-up 5 giây và đo 10 giây. Dùng bộ dữ liệu chuẩn. Mỗi thư mục kết quả chỉ dành cho một lần chạy; không ghi đè kết quả cũ.


In [ ]:
smoke = Path('/kaggle/working/phase2-results-smoke')
run('-m', 'experiments.phase2.runner', '--profile', 'smoke', '--output-root', str(smoke))
run('-m', 'experiments.phase2.report', str(smoke))


## Thí nghiệm chính

Bốn biến thể × bốn kịch bản × ba mức tải × ba lần lặp. Dự kiến 3–4 giờ kể cả chuẩn bị dữ liệu; thời gian thực tế phụ thuộc môi trường. Không đổi source, dependency hoặc cấu hình notebook trong khi chạy. Ô này chỉ chạy tiếp sau khi smoke trả mã thành công.


In [ ]:
full = Path('/kaggle/working/phase2-results-full')
run('-m', 'experiments.phase2.runner', '--profile', 'full', '--output-root', str(full))


## Tổng hợp và xuất kết quả

Nếu phép đo bị ngắt, vẫn có thể chạy ô này riêng để tổng hợp phần đã có; báo cáo giữ trạng thái chưa hoàn tất. Chỉ kết luận Pha 2 khi `primary_complete` là `true`.


In [ ]:
run('-m', 'experiments.phase2.report', str(full))
import json, shutil
summary = json.loads((full / 'summary.json').read_text())
print({k: summary[k] for k in ['expected_runs', 'valid_runs', 'invalid_runs', 'primary_complete']})
print((full / 'report.md').read_text())
shutil.make_archive('/kaggle/working/phase2-results-full', 'gztar', root_dir=full.parent, base_dir=full.name)
